# IS 346: Data Quality - Project
**Prepared by:** Alanoud Alotaibi 445202172 , Najd Altamimi 444201287

**Supervise by:** Dr.Nouf Aldrees

**loading dataset and libraries**

In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re

# 1. Load dataset
path = kagglehub.dataset_download("nudratabbas/messy-clinic-appointments-dataset")
df = pd.read_csv(f"{path}/messy_clinic_appointments.csv")

# 2. Dataset Description
print("DATASET DESCRIPTION")
print("="*50)
desc = """
Attribute            | Description
patient_id           | Unique patient identifier
patient_name         | Full name of the patient
age                  | Age of the patient
gender               | Gender (Male/Female)
appointment_date     | Intended appointment date (dd/mm/yyyy)
booking_date         | Date the appointment was booked
doctor               | Assigned doctor name
department           | Medical department
billing_amount       | Consultation fee (should be numeric)
follow_up_required   | Whether follow-up is needed (Yes/No)
"""
print(desc)

**Data Profiling**

In [ ]:
# 3. Data Profiling
print("DATA PROFILING")

# Shape & columns
print(f"\nShape: {df.shape[0]} rows × {df.shape[1]} columns")
print("Columns:", df.columns.tolist())

# Basic statistics (min, max, average)
print("\n--- Descriptive Statistics ---")
display(df.describe(include='all').T)

# Missing values
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_table = pd.DataFrame({
    'Column': missing.index,
    'Missing Count': missing.values,
    '% Missing': missing_pct.values
}).sort_values('Missing Count', ascending=False)
print("\n--- Missing Values ---")
display(missing_table[missing_table['Missing Count'] > 0])

# Duplicates
exact_dups = df.duplicated().sum()
id_dups = df['patient_id'].duplicated().sum()
print(f"\nExact duplicate rows: {exact_dups}")
print(f"Duplicate patient IDs: {id_dups}")

# Validity issues
# Gender
gender_err = (~df['gender'].isin(['Male','Female']) & df['gender'].notnull()).sum()
print(f"\nGender errors (not strictly Male/Female): {gender_err}")

# Billing amount
numeric_billing = pd.to_numeric(df['billing_amount'], errors='coerce')
billing_non_numeric = (numeric_billing.isnull() & df['billing_amount'].notnull()).sum()
print(f"Billing amount non-numeric: {billing_non_numeric}")

# Age range (0-120)
age_out_of_range = ((df['age'] < 0) | (df['age'] > 120)).sum()
print(f"Age out of 0-120: {age_out_of_range}")

# Appointment date format & future
appt_dates = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y', errors='coerce')
appt_fmt_err = (appt_dates.isnull() & df['appointment_date'].notnull()).sum()
appt_future = (appt_dates > pd.Timestamp.today()).sum()
print(f"Appointment date format errors: {appt_fmt_err}, future dates: {appt_future}")

# Categorical columns – top values
for col in ['gender','doctor','department','follow_up_required']:
    print(f"\n--- {col} top values ---")
    print(df[col].value_counts(dropna=False).head(10))

# Outliers boxplot (Age)
plt.figure(figsize=(8,4))
sns.boxplot(x=df['age'].dropna(), color='lightblue')
plt.title('Age Outliers')
plt.xlabel('Age')
plt.show()

**Classification of Data Quality Issues**

In [ ]:
followup_fail = (~df['follow_up_required'].isin(['Yes','No']) & df['follow_up_required'].notnull()).sum()
expected_depts = ['Cardiology','Neurology','Orthopedics','Pediatrics','General']
dept_fail = (~df['department'].isin(expected_depts) & df['department'].notnull()).sum()

classification_df = pd.DataFrame([
    ["Completeness", "Missing gender values", f"{df['gender'].isnull().sum()} records"],
    ["Completeness", "Missing billing_amount values", f"{df['billing_amount'].isnull().sum()} records"],
    ["Uniqueness", "Duplicate patient IDs (should be unique)", f"{df['patient_id'].duplicated().sum()} duplicates"],
    ["Validity", "Gender not 'Male'/'Female'", f"{(~df['gender'].isin(['Male','Female']) & df['gender'].notnull()).sum()} records"],
    ["Validity", "Billing amount contains text/symbols", f"{(pd.to_numeric(df['billing_amount'], errors='coerce').isnull() & df['billing_amount'].notnull()).sum()} records"],
    ["Validity", "Age outside plausible range (0-120)", f"{((df['age'] < 0) | (df['age'] > 120)).sum()} records"],
    ["Consistency", "Appointment date invalid format", f"{(pd.to_datetime(df['appointment_date'], format='%d/%m/%Y', errors='coerce').isnull() & df['appointment_date'].notnull()).sum()} records"],
    ["Consistency", "Future appointment dates", f"{(pd.to_datetime(df['appointment_date'], format='%d/%m/%Y', errors='coerce') > pd.Timestamp.today()).sum()} records"],
    ["Consistency", "Department values not in expected list", f"{dept_fail} records"],
    ["Consistency", "Follow-up not Yes/No", f"{followup_fail} records"]
], columns=["Category", "Issue Description", "Extent"])

print("CLASSIFICATION OF DATA QUALITY ISSUES")
print("="*70)
display(classification_df)

**Quality Checks Implementation**

In [ ]:
# 5. Quality Checks Implementation
print("QUALITY CHECKS IMPLEMENTATION")
print("="*50)

# 5.1 Completeness (Nulls)
null_counts = df.isnull().sum()

# 5.2 Uniqueness
id_dup_count = df['patient_id'].duplicated().sum()
exact_dup_count = df.duplicated().sum()

# 5.3 Range checks
age_range_fail = ((df['age'] < 0) | (df['age'] > 120)).sum()

billing_num = pd.to_numeric(df['billing_amount'], errors='coerce')
billing_negative = (billing_num < 0).sum()

# 5.4 Format checks (Validity / Consistency)
billing_format_fail = (pd.to_numeric(df['billing_amount'], errors='coerce').isnull() & df['billing_amount'].notnull()).sum()

gender_fail = (~df['gender'].isin(['Male','Female']) & df['gender'].notnull()).sum()

appt_dt = pd.to_datetime(df['appointment_date'], format='%d/%m/%Y', errors='coerce')
date_fail = (appt_dt.isnull() & df['appointment_date'].notnull()).sum() + (appt_dt > pd.Timestamp.today()).sum()

followup_fail = (~df['follow_up_required'].isin(['Yes','No']) & df['follow_up_required'].notnull()).sum()

# Department consistency (example list – adjust as needed)
expected_departments = ['Cardiology','Neurology','Orthopedics','Pediatrics','General']
dept_fail = (~df['department'].isin(expected_departments) & df['department'].notnull()).sum()

# 5.5 Email format check (dataset has no email column)
if 'email' in df.columns:
    email_pattern = r'^[\w\.-]+@[\w\.-]+\.[\w]{2,}$'
    email_fail = (~df['email'].astype(str).str.match(email_pattern) & df['email'].notnull()).sum()
else:
    email_fail = 0
    print("\nNote: No email column in dataset")

# Compile all checks
failures = {
    'Missing Names': 0,  # patient_name has no nulls in this dataset
    'Duplicate IDs': id_dup_count,
    'Age Range (0-120)': age_range_fail,
    'Billing Negative': billing_negative,
    'Billing Format (Currency)': billing_format_fail,
    'Gender Format': gender_fail,
    'Appt Date Invalid': date_fail,
    'Follow-up Format': followup_fail,
    'Department Format': dept_fail,
    'Email Format': email_fail
}

print("\n--- Data Quality Check Summary ---")
failures_series = pd.Series(failures).sort_values(ascending=False)
summary_df = pd.DataFrame({'Check': failures_series.index, 'Failed Records': failures_series.values})
display(summary_df)

# Bar chart
plt.figure(figsize=(12,6))
bars = plt.bar(failures_series.index, failures_series.values, color='#4169E1')
plt.title('Data Quality Monitoring – Failed Records per Check', fontweight='bold')
plt.ylabel('Number of Failed Records')
plt.xticks(rotation=45, ha='right')
plt.grid(axis='y', linestyle='--', alpha=0.5)
for bar in bars:
    h = bar.get_height()
    plt.text(bar.get_x() + bar.get_width()/2., h + 1, int(h),
             ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()